# Supervised Learning: Next-Year FMD Prediction

Regression workflow for next-year frequent mental distress, including target engineering, baselines, tuned models, CV, holdout evaluation, and supervised plots.

This active notebook was split from the original combined learning notebook so the clean repository has clearly delineated supervised and unsupervised work. The original combined notebook is archived under `archive/notebooks/original_combined/`.


## 0. Setup

**Note:** Run the optional install cell only if your notebook environment is missing packages. Also, the core modeling sections expect `scikit-learn`; `UMAP` and `geopandas` are optional because the notebook includes fallbacks.

In [ ]:
# Optional: uncomment and run if your environment is missing these packages.
# %pip install scikit-learn matplotlib seaborn scipy umap-learn geopandas shapely

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

try:
    import umap.umap_ as umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

try:
    from scipy.stats import f_oneway
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

try:
    import geopandas as gpd
    from shapely import wkt
    HAS_GEO = True
except Exception:
    HAS_GEO = False

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

## 1. Load the Imputed County Panel

The dataset is expected at `data/processed/county_panel_imputed.csv` relative to the clean repository root.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "processed" / "county_panel_imputed.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Run this notebook from the clean repository root.")

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}")
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]:,}")
display(raw.head())

In [ ]:
raw.info()

In [ ]:
summary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "n_unique": raw.nunique(dropna=True),
})
display(summary)

print("Years:", sorted(raw["year"].unique()))
print("Unique counties:", raw["geoid"].nunique())
print("Duplicate county-year rows:", raw.duplicated(["geoid", "year"]).sum())

## 2. Target Engineering

We want to see if current-year air quality and socioeconomic conditions can predict **next-year FMD**. We therefore sort by county and year, shift `mental_health_prevalence` one year forward within each county, and drop rows whose following year is unavailable or non-consecutive. We do this to help avoid leakage.

In [ ]:
df = raw.copy()
df["geoid"] = df["geoid"].astype(str).str.zfill(5)
df = df.sort_values(["geoid", "year"]).reset_index(drop=True)

# Shift target and year within each county.
df["next_year"] = df.groupby("geoid")["year"].shift(-1)
df["next_year_fmd"] = df.groupby("geoid")["mental_health_prevalence"].shift(-1)

# Keep only true one-year-ahead prediction rows.
supervised_df = df.loc[df["next_year"].eq(df["year"] + 1)].copy()
supervised_df["next_year"] = supervised_df["next_year"].astype(int)

print(f"Rows available for one-year-ahead supervised modeling: {len(supervised_df):,}")
print(supervised_df.groupby(["year", "next_year"]).size())
display(supervised_df[["geoid", "county_name", "state_name", "year", "mental_health_prevalence", "next_year", "next_year_fmd"]].head())

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(supervised_df["next_year_fmd"], bins=30, kde=True)
plt.title("Distribution of Next-Year Frequent Mental Distress")
plt.xlabel("Next-year FMD prevalence (%)")
plt.ylabel("County-years")
plt.tight_layout()
plt.show()

## 3. Feature Sets

The supervised models use current-year features only. The unsupervised models exclude FMD and FMD confidence intervals, so that the cluster evaluation can treat mental distress as a held-out outcome.

In [ ]:
TARGET = "next_year_fmd"
CURRENT_FMD = "mental_health_prevalence"
ID_COLS = ["geoid", "county_name", "state_name", "year", "next_year", "geometry"]
FMD_RELATED = ["mental_health_prevalence", "lower_ci", "upper_ci", "next_year_fmd"]

# Convert comma-formatted population field if present.
for frame in (supervised_df, df):
    if "total_population" in frame.columns:
        frame["total_population_numeric"] = (
            frame["total_population"].astype(str).str.replace(",", "", regex=False).replace("nan", np.nan).astype(float)
        )

numeric_cols = supervised_df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    c for c in numeric_cols
    if c not in FMD_RELATED + ["next_year"]
]

# `geoid` became string, so it is not in numeric_cols; keep IDs out if a future read infers otherwise.
feature_cols = [c for c in feature_cols if c not in ID_COLS]

air_quality_features = [c for c in feature_cols if any(token in c.lower() for token in ["aqi", "days", "co", "no2", "ozone", "pm2.5", "pm10", "unhealthy", "hazardous", "good", "moderate"])]
socioeconomic_features = [c for c in feature_cols if c not in air_quality_features]

print(f"Model features ({len(feature_cols)}):")
for col in feature_cols:
    print("-", col)

print("\nAir-quality feature count:", len(air_quality_features))
print("Socioeconomic/other feature count:", len(socioeconomic_features))

In [ ]:
feature_overview = supervised_df[feature_cols + [CURRENT_FMD, TARGET]].describe().T
feature_overview["missing"] = supervised_df[feature_cols + [CURRENT_FMD, TARGET]].isna().sum()
display(feature_overview)

## 4. Supervised Learning

### Evaluation design

- **Final holdout:** feature year 2022 predicting FMD in 2023.
- **Training/CV:** earlier feature years, grouped by year where possible.
- **Metrics:** RMSE, MAE, and R².
- **Models:** Dummy baseline, ElasticNet, Random Forest, and Gradient Boosting.

This is stricter than a random split because random county-year splitting can let the model learn information from nearby years for the same counties.

In [ ]:
holdout_feature_year = supervised_df["year"].max()
train_df = supervised_df[supervised_df["year"] < holdout_feature_year].copy()
test_df = supervised_df[supervised_df["year"] == holdout_feature_year].copy()

X_train = train_df[feature_cols]
y_train = train_df[TARGET]
X_test = test_df[feature_cols]
y_test = test_df[TARGET]

groups = train_df["year"]

print(f"Training rows: {len(train_df):,} from feature years {sorted(train_df['year'].unique())}")
print(f"Holdout rows: {len(test_df):,} from feature year {holdout_feature_year}, predicting {int(test_df['next_year'].mode()[0])}")

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_model(name, estimator, X_train, y_train, X_test, y_test):
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    return {
        "model": name,
        "rmse": rmse(y_test, pred),
        "mae": mean_absolute_error(y_test, pred),
        "r2": r2_score(y_test, pred),
        "estimator": estimator,
        "predictions": pred,
    }

models = {
    "Dummy mean baseline": DummyRegressor(strategy="mean"),
    "ElasticNet": Pipeline([
        ("scale", StandardScaler()),
        ("model", ElasticNet(max_iter=50_000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
for name, model in models.items():
    results.append(evaluate_model(name, model, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ["estimator", "predictions"]} for row in results])
results_df = results_df.sort_values("rmse")
display(results_df)

In [ ]:
cv_rows = []
cv = GroupKFold(n_splits=train_df["year"].nunique())
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, groups=groups, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "model": name,
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_rmse_sd": scores["test_rmse"].std(),
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_r2_mean": scores["test_r2"].mean(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("cv_rmse_mean")
display(cv_df)

### Optional Hyperparameter Tuning

This small grid keeps runtime reasonable. Expand the grids if you need a stronger final model for the report.

In [ ]:
elastic_grid = GridSearchCV(
    Pipeline([
        ("scale", StandardScaler()),
        ("model", ElasticNet(max_iter=50_000, random_state=RANDOM_STATE))
    ]),
    param_grid={
        "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.1, 0.5, 0.9],
    },
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={
        "n_estimators": [250, 500],
        "max_features": ["sqrt", 0.7],
        "min_samples_leaf": [1, 3, 8],
    },
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [100, 250],
        "learning_rate": [0.03, 0.07, 0.1],
        "max_depth": [2, 3],
    },
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

tuned_models = {
    "ElasticNet tuned": elastic_grid,
    "Random Forest tuned": rf_grid,
    "Gradient Boosting tuned": gb_grid,
}

tuned_results = []
for name, search in tuned_models.items():
    search.fit(X_train, y_train, groups=groups)
    pred = search.predict(X_test)
    tuned_results.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_rmse": -search.best_score_,
        "holdout_rmse": rmse(y_test, pred),
        "holdout_mae": mean_absolute_error(y_test, pred),
        "holdout_r2": r2_score(y_test, pred),
        "estimator": search.best_estimator_,
        "predictions": pred,
    })

tuned_results_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ["estimator", "predictions"]} for row in tuned_results])
display(tuned_results_df.sort_values("holdout_rmse"))

In [ ]:
all_model_objects = {row["model"]: row for row in results}
all_model_objects.update({row["model"]: row for row in tuned_results})

best_name = min(all_model_objects, key=lambda name: all_model_objects[name].get("holdout_rmse", all_model_objects[name].get("rmse", np.inf)))
best = all_model_objects[best_name]
best_estimator = best["estimator"]
best_pred = best["predictions"]

print("Best holdout model:", best_name)
print("Holdout RMSE:", rmse(y_test, best_pred))
print("Holdout R²:", r2_score(y_test, best_pred))

### Supervised Visualizations

These plots support the report requirements: predicted-vs-actual, residual diagnostics, and feature influence.

In [ ]:
pred_df = test_df[["geoid", "county_name", "state_name", "year", "next_year", TARGET]].copy()
pred_df["prediction"] = best_pred
pred_df["residual"] = pred_df[TARGET] - pred_df["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=pred_df, x=TARGET, y="prediction", alpha=0.65, ax=axes[0])
lims = [min(pred_df[TARGET].min(), pred_df["prediction"].min()), max(pred_df[TARGET].max(), pred_df["prediction"].max())]
axes[0].plot(lims, lims, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Predicted vs. Actual Next-Year FMD")
axes[0].set_xlabel("Actual next-year FMD (%)")
axes[0].set_ylabel("Predicted next-year FMD (%)")

sns.scatterplot(data=pred_df, x="prediction", y="residual", alpha=0.65, ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Residuals by Prediction")
axes[1].set_xlabel("Predicted next-year FMD (%)")
axes[1].set_ylabel("Residual: actual - predicted")

plt.tight_layout()
plt.show()

In [ ]:
def get_feature_importance(estimator, feature_names):
    final = estimator
    if isinstance(estimator, Pipeline):
        final = estimator.named_steps.get("model", estimator.steps[-1][1])

    if hasattr(final, "feature_importances_"):
        values = final.feature_importances_
        label = "importance"
    elif hasattr(final, "coef_"):
        values = final.coef_
        label = "coefficient"
    else:
        return None, None

    return pd.DataFrame({"feature": feature_names, label: values}).sort_values(label, key=lambda s: s.abs(), ascending=False), label

importance_df, importance_label = get_feature_importance(best_estimator, feature_cols)
if importance_df is not None:
    display(importance_df.head(15))
    plt.figure(figsize=(9, 6))
    plot_df = importance_df.head(15).sort_values(importance_label)
    sns.barplot(data=plot_df, x=importance_label, y="feature", color="#4C78A8")
    plt.title(f"Top Feature {importance_label.title()}s: {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} does not expose simple feature importances or coefficients.")

In [ ]:
state_errors = (
    pred_df.groupby("state_name")
    .agg(
        n=("geoid", "count"),
        mean_actual=(TARGET, "mean"),
        mean_prediction=("prediction", "mean"),
        mean_residual=("residual", "mean"),
        rmse=("residual", lambda s: np.sqrt(np.mean(np.square(s)))),
    )
    .sort_values("rmse", ascending=False)
)
display(state_errors.head(15))